In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

import joblib

In [ ]:
data = {
    "attendance": [
        95, 90, 85, 80, 75,
        70, 65, 60, 55, 50,
        98, 92, 88, 78, 68
    ],

    "assignments": [
        100, 95, 90, 85, 80,
        75, 70, 65, 60, 55,
        100, 92, 88, 82, 72
    ],

    "midterm": [
        95, 90, 85, 80, 75,
        70, 65, 60, 55, 50,
        98, 91, 86, 78, 68
    ],

    "quiz": [
        90, 88, 82, 78, 75,
        70, 65, 60, 55, 50,
        95, 89, 84, 76, 66
    ],

    "study_hours": [
        25, 22, 20, 18, 15,
        12, 10, 8, 6, 5,
        28, 23, 19, 16, 11
    ],

    "final_score": [
        95, 91, 86, 81, 76,
        71, 65, 60, 55, 50,
        98, 92, 87, 79, 69
    ]
}


df = pd.DataFrame(data)

df.head()

In [ ]:
X = df[
    [
        "attendance",
        "assignments",
        "midterm",
        "quiz",
        "study_hours"
    ]
]

y = df["final_score"]


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

In [ ]:
model = LinearRegression()

model.fit(
    X_train,
    y_train
)


print("Model training completed successfully")

In [ ]:
y_pred = model.predict(X_test)


mae = mean_absolute_error(
    y_test,
    y_pred
)

r2 = r2_score(
    y_test,
    y_pred
)


print("Mean Absolute Error:", round(mae, 2))

print("R2 Score:", round(r2, 2))

In [ ]:
joblib.dump(
    model,
    "student_performance_model.pkl"
)


print("Model saved successfully")

In [ ]:
from fastapi import FastAPI
import joblib

app = FastAPI(
    title="Student Performance Prediction AI API",
    description="Predict student score using Machine Learning",
    version="1.0"
)


model = joblib.load(
    "student_performance_model.pkl"
)


print("AI Model loaded successfully")

In [ ]:
@app.get("/")
def home():

    return {
        "message": "Hello World"
    }



@app.post("/student-performance")
def student_performance(data: dict):

    input_data = [[
        data.get("attendance"),
        data.get("assignments"),
        data.get("midterm"),
        data.get("quiz"),
        data.get("study_hours")
    ]]

    prediction = model.predict(
        input_data
    )[0]

    if prediction >= 80:
        risk_level = "Low"

    elif prediction >= 50:
        risk_level = "Medium"

    else:
        risk_level = "High"


    return {

        "studentId": data.get("studentId"),

        "predicted_score":
            round(float(prediction), 2),

        "risk_level":
            risk_level,

        "expected_achievement":
            f"{round(float(prediction),2)}%",

        "risk_indicator":
            risk_level
    }

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()


config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="info"
)


server = uvicorn.Server(config)

await server.serve()